# Quantification of shape during growth of propagules

## 1. Documentation and Imports


Created on 01-04-2021

Author: Valentin Laplaud

This code is performing measurements on the propagule contours computed by 'ShapeComputationNB'

In [ ]:
# plotting stuff
import matplotlib as mpl
mpl.use('TkAgg')
%matplotlib inline
import matplotlib.pyplot as plt
from cycler import cycler
#Default colors
colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/30) for i in range(30)]
mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)

# numbers handling
import numpy as np
import pandas as pd

# signal processing 
from scipy.signal import savgol_filter, correlate, correlation_lags
from scipy.interpolate import interp1d

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian, laplace, sobel
from skimage.measure import label, regionprops, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing, remove_small_holes,binary_erosion
from skimage.color import rgb2gray
from skimage.segmentation import active_contour, morphological_geodesic_active_contour,morphological_chan_vese, checkerboard_level_set, inverse_gaussian_gradient
import cv2 as cv

# to hide known warnings
import warnings

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf


## 2. Define analysis functions

### 2.0 Change reference from time to area

Create a CD dataframe in which the 'img' column that is used to average data is now in correspondance to the area of each propagule instead of time.  


In [ ]:
def time2area(GD,CD,StackList,nbin):
    
    warnings.filterwarnings("ignore")  
    
    # Compute bins over all areas
    n,bins = np.histogram(GD['Area'],bins=nbin)
       
    # Adds the area group for global dataframe
    GD['AreaGroup'] = pd.cut(GD['Area'],bins,labels=np.arange(1,nbin+1)).to_numpy()
    
    for s in StackList:
   
        nimg = int(1+np.max(CD.loc[s, 'Img']))
        
        for i in range(nimg):

            CD.loc[(CD.index==s) &(CD['Img']==i),'AreaGroup'] = GD.loc[
                (GD.index==s) &(GD['Img']==i),'AreaGroup']
            
    warnings.filterwarnings("default")  
    
    return(GD,CD)

### 2.1 Mean contour over all propagule

#### 2.1.2 In time

Computation only for the number of images of the shortest video. Ponctually one gemma can be ignored for one time point if the focus on the video is bad.

In [ ]:
def computeMeanContourTime(StackList,CD,GD):
    
    # Number of image in the shortest videos
    nmin = 1 + int(np.min([np.max(CD.loc[s, 'Img']) for s in StackList]))
    
    Theta = CD[CD['Img']==0].loc[StackList[0],'Theta'].to_numpy()
    
    # Storing variables
    alignedRs = np.empty((len(Theta),len(StackList)))
    alignedRs[:] = np.nan
    
    for i in range(nmin):
        
        print('Computing mean contour for image : ' + str(i) + '/' + str(nmin).ljust(5), end='\r')
        
        # Not normalized
        
        # On each image, retrieve aligned contour for each propagule
        
        for ii,s in zip(range(len(StackList)),StackList):
           
            rn = CD.loc[(CD.index == s) & (CD['Img'] == i),'alignR']            
            
            if bool(GD.loc[(GD.index == s) &
                                   (GD['Img'] == i),'GoodFocus'].to_numpy) == True:
                alignedRs[:,ii] = rn

        # Computing mean of aligned contours        
        meanR = np.nanmean(alignedRs,1) 
        
        meanX,meanY = vf.ToCart(Theta, meanR, angle='deg')
        
        # Storing contour data
        data = {'X':meanX,
                    'Y':meanY,  
                    'alignR':meanR,  
                    'Theta':Theta,
                    'Img':i*np.ones(len(meanX))} 
      
        newdata = pd.DataFrame(data=data,index = np.repeat('meanContTime',len(meanX)))
        
        CD = CD.append(newdata) # adding to global dataframe
        
        # Normalized
        
        # On each image, retrieve aligned contour for each propagule
        
        for ii,s in zip(range(len(StackList)),StackList):
           
            rn = CD.loc[(CD.index == s) & (CD['Img'] == i),'alignRnorm']            
            
            if bool(GD.loc[(GD.index == s) &
                                   (GD['Img'] == i),'GoodFocus'].to_numpy) == True:
                alignedRs[:,ii] = rn

        # Computing mean of aligned contours        
        meanR = np.nanmean(alignedRs,1) 
        
        meanX,meanY = vf.ToCart(Theta, meanR, angle='deg')
        
        # Storing contour data
        data = {'X':meanX,
                    'Y':meanY,  
                    'alignR':meanR,  
                    'Theta':Theta,
                    'Img':i*np.ones(len(meanX))} 
      
        newdata = pd.DataFrame(data=data,index = np.repeat('normMeanContTime',len(meanX)))
        
        CD = CD.append(newdata) # adding to global dataframe

    print('Computing mean contour for image : ' + str(nmin) + '/' + str(nmin).ljust(5), end='\n\n')
        
    return(CD)

#### 2.1.2 In area

Computation of mean contour with different contour grouped based on area rather than timepoint. When several timepoints of the same propagules are in the same area group they are first averaged together before being averaged with the rest of the propagules.

In [ ]:
def computeMeanContourArea(StackList,CD,GD):
    
    # Number of area groups
    nmax = 1 + int(np.max([np.max(CD.loc[s, 'AreaGroup']) for s in StackList]))
        
    Theta = CD[CD['Img']==0].loc[StackList[0],'Theta'].to_numpy()
    
    
    
    # Storing variables
    alignedRs = np.empty((len(Theta),len(StackList)))
    alignedRs[:] = np.nan
    
    for i in range(1,nmax+1):
        
        print('Computing mean contour for area group : ' + str(i) + '/' + str(nmax).ljust(5), end='\r')
        
        # On each image, retrieve aligned contour for each propagule
        
        for ii,s in zip(range(len(StackList)),StackList):
           
            imglist = np.unique(CD.loc[(CD.index == s) & (CD['AreaGroup'] == i),'Img'].to_numpy())

            if not imglist.size==0:
                
                AreaRs = np.empty((len(Theta),len(imglist)))
                AreaRs[:] = np.nan
            
                for iii,img in zip(range(len(imglist)),imglist): 
                    
                    rn = CD.loc[(CD.index == s) & (CD['Img'] == img),'alignR']

                    AreaRs[:,iii] = rn
                
                alignedRs[:,ii] = np.nanmean(AreaRs,1)

        # Computing mean of aligned contours        
        meanR = np.nanmean(alignedRs,1) 
        
        meanX,meanY = vf.ToCart(Theta, meanR, angle='deg')
        
        # Storing contour data
        data = {'X':meanX,
                    'Y':meanY,  
                    'alignR':meanR,  
                    'Theta':Theta,
                    'AreaGroup':i*np.ones(len(meanX))} 
      
        newdata = pd.DataFrame(data=data,index = np.repeat('meanContArea',len(meanX)))
        
        CD = CD.append(newdata) # adding to global dataframe

    print('Computing mean contour for areagroup : ' + str(nmax) + '/' + str(nmax).ljust(5), end='\n\n')
        
    return(CD)

### 2.2 Computation of growth and similarity 

Similarity : 
Comparison of one normalized contour in time rn(theta,t) with either the first contour of the same gemma rn0(theta,t) - selfSimilarity- or with the average contour r(theta,t) -meanSimilarity-.

Growth : 
Percentage of radial extension Gn(theta, t)

In [ ]:
def computeGrowthAndSimi(StackList,CD,GD):  
       
    # define storing variables
    CD['Growth'] = np.nan    
    GD['meanSimi'] = np.nan
    GD['selfSimi'] = np.nan    
       
    # Number of image in the shortest and longest videos
    tmax = 1 + int(np.max([np.max(CD.loc[s, 'Img']) for s in StackList]))
    tmin = 1 + int(np.min([np.max(CD.loc[s, 'Img']) for s in StackList]))
    
    
    Theta = CD.loc[(CD.index == StackList[0]) & (CD['Img'] == 0),'Theta']
        
    GrowthMat = np.empty((len(StackList),len(Theta),tmax)) # Growth Gn(Theta,t) 
        
    for ni,n in zip(range(len(StackList)),StackList):
        
        print('Processing ' + n + ' :')
            
        timg = int(1 + np.max(CD.loc[n, 'Img'])) # number of images
        
        for ti in range(timg):
            
            print('Computing similarities and growth for image : ' + str(ti+1) + '/' + str(timg).ljust(5), end='\r')
            
            # Normalized radius and start (t=0) radius
            Rn = CD.loc[(CD.index == n) & (CD['Img'] == ti),'alignRnorm'].to_numpy()
            Rn0 = CD.loc[(CD.index == n) & (CD['Img'] == 0),'alignRnorm'].to_numpy()
            
            # Computing self similarity
            selfSimi = np.sqrt(np.mean(np.square(Rn-Rn0)))
            GD.loc[(GD.index == n) & (GD['Img'] == ti),'selfSimi'] = selfSimi
            
            # Radius, start raidus, and mean radius
            rn = CD.loc[(CD.index == n) & (CD['Img'] == ti),'alignR'].to_numpy()
            rn0 = CD.loc[(CD.index == n) & (CD['Img'] == 0),'alignR'].to_numpy()
            r = CD.loc[(CD.index == 'meanContTime') & (CD['Img'] == ti), 'alignR'].to_numpy()

            if not ti >= tmin:
                # Computing mean similarity

                
                meanSimi = np.sqrt(np.mean(np.square(rn-r))) 

                GD.loc[(GD.index == n) & (GD['Img'] == ti),'meanSimi'] = meanSimi
            
                
            # Growth
            Gn = (np.divide(rn,rn0)-1)*100
            
            CD.loc[(CD.index == n) & (CD['Img'] == ti),'Growth'] = Gn
            
            GrowthMat[ni,:,ti] = Gn
        
        print('Computing similarities and growth for image : ' + str(ti+1) + '/' + str(timg).ljust(5), end='\n\n')    
            
    return(CD,GD,GrowthMat)         
            

### 2.3 Estimating symmetry of the shape

#### 2.3.0 Symmetry computation

Computes the symmetry of a radially define contour, with an axis of symmetry going through Theta0 -> Theta0+180

In [ ]:
def Sym(Theta,R,Theta0, **kwargs):
    
    DebugPlots = False
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        else:
            print('Unknown key : ' + key+ '. Kwarg ignored.')
    
    npts = len(Theta)
    i0 = np.argmin(np.absolute(Theta-Theta0))+1
    i180 = int(i0+npts/2)
    
    R1 = np.roll(R,-i0)[0:int(npts/2-1)]
    R2 = np.flip(np.roll(R,-i180)[0:int(npts/2-1)])
    
    if DebugPlots:
        plt.figure(dpi=250)
        ax1 = plt.subplot(111, polar=False)
        ax1.plot(np.deg2rad(Theta),np.roll(R,-i0),'r')
        ax1.plot(np.deg2rad(Theta[0:int(npts/2-1)]),R1,'c--',lw=0.8)
        ax1.plot(np.deg2rad(Theta[0:int(npts/2-1)]),R2,'g--',lw=0.8)
        ax1.set_yticklabels(['','','','','',''])
        plt.show()

    aSym = np.sqrt(np.mean(np.square(R1-R2)))
    
    return(aSym)

#### 2.3.1 Compute symmetry for all contours

In [ ]:
def computeSym(StackList,CD,GD, **kwargs):    
      
    DebugPlots = False
    
    for key, value in kwargs.items(): 
        if key == 'debug':
            DebugPlots = value
        else:
            print('Unknown key : ' + key+ '. Kwarg ignored.')
        
    GD['Assymmetry'] = np.nan
    
    for n in StackList:
        
        timg = int(1 + np.max(CD.loc[n, 'Img'])) # number of images
        
        for ti in range(timg):
            
            Theta = CD.loc[(CD.index == n) & (CD['Img'] == ti),'Theta'].to_numpy()
            rn = CD.loc[(CD.index == n) & (CD['Img'] == ti),'alignR'].to_numpy()
            
            aSym = Sym(Theta,rn,0,debug=DebugPlots)
            
            GD.loc[(GD.index == n) & (GD['Img'] == ti),'Assymmetry'] = aSym
            
    return(CD,GD)

## 3. Plot fuctions

### 3.1 Plot propagules area in time

In [ ]:
def plotArea(StackList, Data, P, Title, **kwargs):

    # Colors
    colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/30) for i in range(30)]
    mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)
    
    #init and read kwargs
    Trap = False # default value
    soloGraphs = False
    
    for key, value in kwargs.items(): 
        if key == 'Trapped':
            Trap = value
        elif key == 'solo':
            soloGraphs = value
        else:
            print('Unknown key : ' + key+ '. Kwarg ignored.')
         
    # check existence of figure folder, if absent, create it
    if not os.path.exists(P + '\\Figures'):
            os.mkdir(P + '\\Figures') # create folder
    
    # plot figure
    
    ###################### plot individual graphs #######################
    
    if soloGraphs: 
    
        for s in StackList:
            

            fig1 = plt.figure(dpi = 200)
            fig1.suptitle(s + ' - Size vs. time')
            plt.xlabel('Time (min)')
            plt.ylabel('Size (mm²)')
            

            xTime = Data.at[s, 'Time (min)'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        
            
            PropBBOX = Data.at[s, 'Bounding box (px)']

            plt.plot(xTime,Data.at[s, 'Size (mm²)'].to_numpy()[Data.at[s, 'GoodFocus']==True],'-o',ms=3,figure=fig1,label=s)

            # if there is a trap, find out when th ppgls touches up and bottom wall
            if Trap:
                trapdata =vf.LoadImageJResults(P + '\\' + s + '_Trap.txt', ['BX','BY','Width','Height'])
                TrapBBOX = [trapdata.at[0, 'BY'], trapdata.at[0, 'BX'],trapdata.at[0, 'BY'] + trapdata.at[0, 'Height'] ,
                        trapdata.at[0, 'BX'] + trapdata.at[0, 'Width']]

                BBOXDiff = PropBBOX - TrapBBOX            
                idxTouch = np.argwhere(np.logical_and(BBOXDiff[0:,0]<0, BBOXDiff[0:,2]>0))

                plt.plot(xTime[idxTouch],Data.at[s, 'Size (mm²)'][idxTouch],'-or',lw=0.5,ms=1.5,figure=fig1,label='Touching UpDown')
                plt.legend(prop={'size': 4})

                fig1.savefig(P + '\\Figures\\' + s + '_SizeTime.png')    


    ###################### plot common graphs #######################
    
    fig2,ax2 = plt.subplots(dpi = 200)
    fig2.suptitle(Title + ' - Area vs. time')
    plt.xlabel('Time (min)')
    plt.ylabel('Area (µm²)')
    
    fig3,ax3 = plt.subplots(dpi = 200)
    fig3.suptitle(Title + ' - Norm Area vs. time')
    plt.xlabel('Time (min)')
    plt.ylabel('Area (normalized)')
    plt.legend(prop={'size': 4})
    
    for s in StackList:
        
                   
        xTime = Data.at[s, 'Time (min)'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        Area = Data.at[s, 'Area'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        
        NormArea = Area/Area[0]
        
        ax2.plot(xTime,Area,'-o',ms=3,label=s)
        ax3.plot(xTime,NormArea,'-o',ms=3,label=s)
        
    
    plt.figure(fig2.number)
    plt.legend(prop={'size': 4})
    fig2.savefig(P + '\\Figures\\AreaTime_' + Title + '.png')
    
    plt.figure(fig3.number)
    plt.legend(prop={'size': 4})
    fig3.savefig(P + '\\Figures\\NormAreaTime_' + Title + '.png')
    
    
        

### 3.2 Plot evolution of propagule shape in time and mean propagule contour

In [ ]:
def plotShapes(P, StackList,CD,GD, figname):
    
    # create save folders
    if not os.path.exists(P + '\\Figures\\XYshape\\'):
            os.mkdir(P + '\\Figures\\XYshape\\') 
            
    if not os.path.exists(P + '\\Figures\\ThetaR\\'):
            os.mkdir(P + '\\Figures\\ThetaR\\') 
                    
    if not os.path.exists(P + '\\Figures\\MeanContourTimeNorm_' + figname + '\\'):
            os.mkdir(P + '\\Figures\\MeanContourTimeNorm_' + figname + '\\') 
            
    
    # Number of images
    nmint = 1 + int(np.min([np.max(CD.loc[s, 'Img']) for s in StackList]))
    narea = 1 + int(np.max([np.max(CD.loc[s, 'AreaGroup']) for s in StackList]))
        
    # Colors
    colorcycle = [plt.get_cmap('jet')(1. * i/nmint) for i in range(nmint)]
    mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)


    ################ Mean Contours in time #################
    
    # initialize figure for mean contour
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('MeanContour in time - ' + figname)
    plt.xlabel('(µm)')
    plt.ylabel('(µm)')
    ax0.plot(0,0,'*r')                               
    ax0.set_aspect('equal', adjustable='box')
    
    print('Plotting mean contours in time...', end='')
    
    for i in range(nmint):
  
        # Plot shape
        meanX = CD.loc[(CD.index == 'meanContTime') & (CD['Img'] == i),'X']  
        meanY = CD.loc[(CD.index == 'meanContTime') & (CD['Img'] == i),'Y']
        
        ax0.plot(meanX,meanY,'-', lw=0.5)
            
    # Save fig
    fig0.savefig(P + '\\Figures\\MeanContourTime_' + figname + '.png')
    plt.figure(fig0.number)
    plt.close(fig0.number)
    
    print('Done and saved.')
    
    
    print('Plotting normalized mean contours in time...', end='')
    
    colorcyclejet = [plt.get_cmap('jet')(1. * i/nmint) for i in range(nmint)]
    
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('Normalized meanContour in time - ' + figname)
    plt.xlabel('(norm)')
    plt.ylabel('(norm)')
    plt.xlim(-1.1,1.1)
    plt.ylim(-1.1,1.1)
    ax0.plot(0,0,'*r')                               
    ax0.set_aspect('equal', adjustable='box')    
    
    for i in range(nmint):
    
        # initialize figue for normalized mean contour
        fig1, ax1 = plt.subplots()
        fig1.set_dpi(250)
        fig1.suptitle('Normalized meanContour in time - ' + figname)
        plt.xlabel('(norm)')
        plt.ylabel('(norm)')
        plt.xlim(-1.1,1.1)
        plt.ylim(-1.1,1.1)
        ax1.plot(0,0,'*r')                               
        ax1.set_aspect('equal', adjustable='box')     
  
        # Plot shape
        meanX = CD.loc[(CD.index == 'normMeanCont') & (CD['Img'] == i),'X']  
        meanY = CD.loc[(CD.index == 'normMeanCont') & (CD['Img'] == i),'Y']
        
        ax0.plot(meanX,meanY,'-', lw=1.5, c = colorcyclejet[i])
        ax1.plot(meanX,meanY,'-', lw=1.5, c = colorcyclejet[i])
            
        # Save fig
        fig1.savefig(P + '\\Figures\\MeanContourTimeNorm_' + figname + '\\MeanContourTimeNorm_' + figname + '_' + str(i) + '.png')
        plt.figure(fig1.number)
        plt.close(fig1.number)
        
    # Save fig
        fig0.savefig(P + '\\Figures\\MeanContourTimeNormAll_' + figname + '.png')
        plt.figure(fig0.number)
        plt.close(fig0.number)
    
    print('Done and saved.')
    
    
    ################ Mean Contours in area #################
    
    # initialize figure for mean contour
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('MeanContour in area - ' + figname)
    plt.xlabel('(µm)')
    plt.ylabel('(µm)')
    ax0.plot(0,0,'*r')                               
    ax0.set_aspect('equal', adjustable='box')
    
    print('Plotting mean contours in area...', end='')
    
    
    colorcyclejet = [plt.get_cmap('jet')(1. * i/narea) for i in range(narea)]
    
    for i in range(narea):
  
        #Plot shape
        meanX = CD.loc[(CD.index == 'meanContArea') & (CD['AreaGroup'] == i),'X']  
        meanY = CD.loc[(CD.index == 'meanContArea') & (CD['AreaGroup'] == i),'Y']
        
        ax0.plot(meanX,meanY,'-', lw=0.5,c=colorcyclejet[i])
            
    # Save fig
    fig0.savefig(P + '\\Figures\\MeanContourArea_' + figname + '.png')
    plt.figure(fig0.number)
    #plt.close(fig0.number)
    
    print('Done and saved.')
        
    ################ Individual Contours #################
                  
    for s in StackList:

        # intialize figures
        fig1, ax1 = plt.subplots(dpi = 250)
        fig1.suptitle(figname + ' - ' + s)
        plt.xlabel('(µm)')
        plt.ylabel('(µm)')
        ax1.plot(0,0,'*r')                               
        ax1.set_aspect('equal', adjustable='box')

        fig2, ax2 = plt.subplots(dpi = 250)
        fig2.suptitle(figname + ' - ' + s)
        plt.xlabel('Angle (°)')
        plt.ylabel('Radius (µm)')

        n = int(1 + np.max(CD.loc[s, 'Img']))
        
        colorcycle = [plt.get_cmap('jet')(1. * i/n) for i in range(n)]
        mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)
            
        print('Plotting individual contours for gemma : ' + s + '...', end='')
        for i in range(n):

                if (GD.loc[s]['GoodFocus'].to_numpy()[i] == True):

                    A = CD[CD['Img']==i].loc[s,'Theta']
                    R = CD[CD['Img']==i].loc[s,'alignR']    

                    X,Y = vf.ToCart(A,R, angle = 'deg') 

                    ax1.plot(X,Y,'-', ms = 0.1, lw = 0.5)
                    ax2.plot(A,R,'-', ms = 0.1, lw = 0.5)
                    

        fig1.savefig(P + '\\Figures\\XYshape\\' + s + '_AligShape.png')
        plt.close(fig1.number)
        fig2.savefig(P + '\\Figures\\ThetaR\\' + s + '_AligRadius.png') 
        plt.figure(fig2.number)
        plt.close(fig2.number)
        
        print('Done and saved.')
 

### 3.3 Plot shape characteristics

#### 3.3.1 Plot similarities in time

In [ ]:
def plotShapeSimi(P,StackList,CD,GD, figname):
    
    # Number of image in the shortest and longest videos
    nmin = 1 + int(np.min([np.max(CD.loc[s, 'Img']) for s in StackList]))
    nmax = 1 + int(np.max([np.max(CD.loc[s, 'Img']) for s in StackList]))
    
    ### Area and selfSimilarity comparison in time
    
    # initialize figure
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('selfSimi vs. Area - ' + figname)
    plt.xlabel('Area (norm)')
    plt.ylabel('Similarity to self')                
    
    print('Plot self similarity graphs...', end='')
    
    for s in StackList:
        
        xTime = GD.at[s, 'Time (min)'].to_numpy()[GD.at[s, 'GoodFocus']==True]/60
        Area = GD.at[s, 'Area'].to_numpy()[GD.at[s, 'GoodFocus']==True]        
        NormArea = Area/Area[0]        
        selfSimi = GD.at[s, 'selfSimi'].to_numpy()[GD.at[s, 'GoodFocus']==True]

        ax0.plot(NormArea,selfSimi,'k-', lw = 0.5,zorder=1)
        sc = ax0.scatter(NormArea,selfSimi, c= xTime, cmap = 'jet', vmin=0, vmax = 0.5*nmax,zorder=5)
    
    fig0.colorbar(sc, ax = ax0, label = 'Time (h)')
    
    fig0.savefig(P + '\\Figures\\AreaSelfSimi.png')
    
    print('Done and saved.')
    

    ### Area and meanSimilarity comparison in time
    
    # initialize figure
    fig1, ax1 = plt.subplots()
    fig1.set_dpi(250)
    fig1.suptitle('meanSimi vs. Area - ' + figname)
    plt.xlabel('Area (norm)')
    plt.ylabel('Similarity to mean')                
    
    print('Plot mean similarity graphs...', end='')
    
    for s in StackList:
        
        xTime = GD.at[s, 'Time (min)'].to_numpy()[GD.at[s, 'GoodFocus']==True]/60
        Area = GD.at[s, 'Area'].to_numpy()[GD.at[s, 'GoodFocus']==True]        
        NormArea = Area/Area[0]        
        meanSimi = GD.at[s, 'meanSimi'].to_numpy()[GD.at[s, 'GoodFocus']==True]
        
        ax1.plot(NormArea,meanSimi,'k-', lw = 0.5,zorder=1)
        sc = ax1.scatter(NormArea,meanSimi, c= xTime, cmap = 'jet', vmin=0, vmax = 0.5*nmax,zorder=5)
    
    fig1.colorbar(sc, ax = ax1, label = 'Time (h)')
    
    fig1.savefig(P + '\\Figures\\AreaMeanSimi.png')
    
    print('Done and saved.')

#### 3.3.2 Plot Growth quantifications

Growth Gn(Theta,t) = G(n,Th,t)

In [ ]:
def plotGrowth(P,StackList,CD,GD,GrowthMat):
    
    ### Individual growth polar plot    
    
    # Specifics of circular histogram
    N = 20 # number of quadrant
    theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False) # Abcisse
    width = (2*np.pi) / N # bar width    
    bottom = 0 # Bottom level
    
    if not os.path.exists(P + '\\Figures\\Growth\\'):
        os.mkdir(P + '\\Figures\\Growth\\') # create hue image folder   
    
    A = CD[CD['Img']==0].loc[StackList[0],'Theta'].values+180
    
    for s in StackList:    
        
        print('Plot growth for gemma : ' + s + '...', end='')
        
        if not os.path.exists(P + '\\Figures\\Growth\\' + s  + '_Growth\\'):
            os.mkdir(P + '\\Figures\\Growth\\' + s  + '_Growth\\') # create save folder
       
        n = int(1 + np.max(CD.loc[s, 'Img']))
        
        for i in range(n):
            
            
            if (GD.loc[s]['GoodFocus'].to_numpy()[i] == True):
                
 
                G = CD[CD['Img']==i].loc[s,'Growth']

                # initialize figure
                fig1 = plt.figure(dpi = 250)
                fig1.suptitle(s + ' growth - T = ' +str(GD[GD['Img']==i].loc[s,'Time (min)']))
                ax1 = plt.subplot(111, polar=True)
                ax1.set_ylim(ymin = -1.5, ymax = 100)
                ax1.set_yticks([0,25,50,75,100])
                ax1.set_yticklabels(['','','50%','','100%'])
                
                radii = np.empty(len(theta))
                # Compute hist values
                radii[1:] = [G[(A > np.mod(np.rad2deg(th-width/2),360)) & (A < np.mod(np.rad2deg(th+width/2),360))].mean() for th in theta[1:]]
                radii[0] = G[(A > np.mod(np.rad2deg(theta[0]-width/2),360)) | (A < np.mod(np.rad2deg(theta[0]+width/2),360))].mean()
                
                bars = ax1.bar(theta, radii, width=width, bottom=bottom)

                # Use custom colors
                for r, bar in zip(radii, bars):
                    bar.set_facecolor(plt.cm.jet(r / 100.))
                
                fig1.savefig(P + '\\Figures\\Growth\\' + s  + '_Growth\\' + str(i) + '.png') 
                
                plt.close(fig1.number)
                
                del G
                
        print('Done and saved.\n')
    
    print('Plotting average growth...')

    if not os.path.exists(P + '\\Figures\\Growth\\' + 'MeanGrowth\\'):
            os.mkdir(P + '\\Figures\\Growth\\' + 'MeanGrowth\\') # create save folder

    nmax = 1 + int(np.max([np.max(CD.loc[s, 'Img']) for s in StackList]))
   

    for i in range(nmax):
        # initialize figure
        fig1 = plt.figure(dpi = 250)
        fig1.suptitle('Meangrowth - T = ' + str(GD[GD['Img']==i].loc[StackList[0],'Time (min)']))
        ax1 = plt.subplot(polar=True)
        ax1.set_ylim(ymin = -1.5, ymax = 100)
        ax1.set_yticks([0,25,50,75,100])
        ax1.set_yticklabels(['','','50%','','100%'])

        
        radii = np.empty(len(theta))
        # Compute hist values
        radii[1:] = [np.nanmean(GrowthMat[:,(A > np.mod(np.rad2deg(th-width/2),360)) &
                           (A < np.mod(np.rad2deg(th+width/2),360)),i]) for th in theta[1:]]
        
        radii[0] = GrowthMat[:,(A > np.mod(np.rad2deg(theta[0]-width/2),360)) |
                             (A < np.mod(np.rad2deg(theta[0]+width/2),360)),i].mean()
        
        bars = ax1.bar(theta, radii, width=width, bottom=bottom)
        
        # Use custom colors
        for r, bar in zip(radii, bars):
            bar.set_facecolor(plt.cm.jet(r / 100.))

        fig1.savefig(P + '\\Figures\\Growth\\' + 'MeanGrowth\\' + str(i) + '.png') 

        plt.close(fig1.number)

    

#### 3.3.3 Plot assymetry

In [ ]:
def plotAssymetry(P, StackList,GD):
    
    colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/len(StackList)) for i in range(len(StackList))]
    mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)
    
   # Assymmetry in time
    fig0, ax0 = plt.subplots(dpi=250)
    fig0.suptitle('Assymmetry in time')     

    # Assymmetry vs area
    fig1, ax1 = plt.subplots(dpi=250)
    fig1.suptitle('Assymmetry vs area')   
    
    for s in StackList:
        
        xTime = GD.loc[s,'Time (min)']
        Assym = GD.loc[s,'Assymmetry']
        Area = GD.loc[s,'Area']
            
        ax0.plot(xTime,Assym,'-o',ms = 3)
        
        ax1.plot(Area,Assym,'-o', ms = 3)
        
    fig0.legend(StackList,prop={'size': 5})
    fig1.legend(StackList,prop={'size': 5})
    
    fig0.savefig(P + '\\Figures\\AssymTime.png') 
    fig1.savefig(P + '\\Figures\\AssymArea.png') 

    

## 4. Enter data and run

### 4.1 Data details

In [ ]:
# Paths to data
P1 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.18'

P2 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.19'


# List of stacks to analyse

# experiments of 18-03-2021
StackList210318 = ['PPG1','PPG2','PPG3','PPG4','PPG6','PPG7','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']

            # No Rhizoid
StackList210318 = ['PPG2','PPG3','PPG6','PPG7','PPG9','PPG10','PPG11',
                   'PPG14','PPG15','PPG16','PPG18','PPG19',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']      

#StackList210318 = ['PPG2'] # Tests



# experiments of 19-03-2021 (68h, cropped when air arrives, at least 52 images)
StackList210319 = ['PPG1','PPG2','PPG6','PPG8','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22']



#StackList210319 = ['PPG17','PPG1'] # Tests


### 4.2 Loading data 

In [ ]:
## Expe 210318
ContourData210318 = pd.read_csv(P1 + '\\ContourData210318.csv', index_col = 'Ind')
GlobalData210318 = pd.read_csv(P1 + '\\GlobalData210318.csv', index_col = 'Ind')    

## Expe 210319
ContourData210319 = pd.read_csv(P2 + '\\ContourData210319.csv', index_col = 'Ind')
GlobalData210319 = pd.read_csv(P2 + '\\GlobalData210319.csv', index_col = 'Ind')

### 4.3 Compute contour characteristics

#### 4.3.0 Ordering contours based on area 

In [ ]:
GlobalData210318,ContourData210318  = time2area(GlobalData210318,ContourData210318,StackList210318,40)

GlobalData210319,ContourData210319  = time2area(GlobalData210319,ContourData210319,StackList210319,120)

#### 4.3.1 Mean contours

In [ ]:
## Expe 210318
print('\033[1m' + '\033[4m' + '\nAnalyzing experiments from 2021-03-18 :\n' + '\033[0m')
ContourData210318 = computeMeanContourTime(StackList210318,ContourData210318,GlobalData210318)
ContourData210318 = computeMeanContourArea(StackList210318,ContourData210318,GlobalData210318)

## Expe 210319
print('\033[1m' + '\033[4m' + 'Analyzing experiments from 2021-03-19 :\n' + '\033[0m')
ContourData210319 = computeMeanContourTime(StackList210319,ContourData210319,GlobalData210319)
ContourData210319 = computeMeanContourArea(StackList210319,ContourData210319,GlobalData210319)


#### 4.3.2 Compute similarities and growth

In [ ]:
## Expe 210318
print('\033[1m' + '\033[4m' + '\nAnalyzing experiments from 2021-03-18 :\n' + '\033[0m')
ContourData210318,GlobalData210318,GrowthMat210318 = computeGrowthAndSimi(StackList210318,ContourData210318,GlobalData210318)

## Expe 210319
print('\033[1m' + '\033[4m' + 'Analyzing experiments from 2021-03-19 :\n' + '\033[0m')
ContourData210319,GlobalData210319,GrowthMat210319 = computeGrowthAndSimi(StackList210319,ContourData210319,GlobalData210319)

#### 4.3.3 Compute assymetry

In [ ]:
ContourData210318,GlobalData210318 = computeSym(StackList210318,ContourData210318,GlobalData210318)

ContourData210319,GlobalData210319 = computeSym(StackList210319,ContourData210319,GlobalData210319)

#### 4.3.5 Save quantifications

In [ ]:
## Saving
GlobalData210318.to_csv(P1 + '\\GlobalData210318_Quantif.csv',index_label = 'Ind')
ContourData210318.to_csv(P1 + '\\ContourData210318_Quantif.csv',index_label = 'Ind')

GlobalData210319.to_csv(P2 + '\\GlobalData210319_Quantif.csv',index_label = 'Ind')
ContourData210319.to_csv(P2 + '\\ContourData210319_Quantif.csv',index_label = 'Ind')

### 4.4 Plot results

#### 4.4.1 Load quantifications

In [ ]:
## Expe 210318
ContourData210318 = pd.read_csv(P1 + '\\ContourData210318_Quantif.csv', index_col = 'Ind')
GlobalData210318 = pd.read_csv(P1 + '\\GlobalData210318_Quantif.csv', index_col = 'Ind')    

## Expe 210319
ContourData210319 = pd.read_csv(P2 + '\\ContourData210319_Quantif.csv', index_col = 'Ind')
GlobalData210319 = pd.read_csv(P2 + '\\GlobalData210319_Quantif.csv', index_col = 'Ind')

#### 4.4.2 plots

In [ ]:
print('\033[1m' + '\033[4m' + 'Plotting for experiments from 2021-03-18 :\n' + '\033[0m')

plotArea(StackList210318, GlobalData210318, P1, 'Expe_210318')

plotShapes(P1, StackList210318,ContourData210318,GlobalData210318, 'Expe_210318')

plotShapeSimi(P1, StackList210318,ContourData210318,GlobalData210318, 'Expe_210318')

plotGrowth(P1,StackList210318,ContourData210318,GlobalData210318,GrowthMat210318)

plotAssymetry(P1, StackList210318,GlobalData210318)

print('\n\n')

print('\033[1m' + '\033[4m' + 'Plotting for experiments from 2021-03-19 :\n' + '\033[0m')

plotArea(StackList210319, GlobalData210319, P2, 'Expe_210319')

plotShapes(P2, StackList210319,ContourData210319,GlobalData210319, 'Expe_210319')

plotShapeSimi(P2, StackList210319,ContourData210319,GlobalData210319, 'Expe_210319')

plotGrowth(P2,StackList210319,ContourData210319,GlobalData210319,GrowthMat210319)

plotAssymetry(P2, StackList210319,GlobalData210319)

print('\n\n')

# Test Zone

In [ ]:

plotShapes(P2, StackList210319,ContourData210319,GlobalData210319, 'Expe_210319')

In [ ]:
plt.figure(dpi=250)
GlobalData210318['Area'].hist(bins=40,rwidth=0.9,color='r')
plt.grid(b=None)
plt.xlabel('Area (µm²)')